# Splitter Test Notebook
In diesem Notebook wird geprüft, ob die Funktion `find_valid_phrases_list` alle relevanten Text-Spans aus der `test.json` erkennt.

In [16]:
import json
import sys
import os

# Add the schema directory to the path so we can import the splitter
sys.path.append(os.path.abspath("schema"))

from splitter import find_valid_phrases_list


In [ ]:
test_json_path = "../../phee-with-chatgpt/data/converted_data/text2spotasoc/event/phee2_cross1/test.json"

with open(test_json_path, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

print(f"Geladene Beispiele: {len(test_data)}")

Geladene Beispiele: 968


In [18]:
results = []
missing_spans_report = []

for example in test_data:
    text = example["text"]
    
    # Collect all labeled spans from all events and their arguments
    label_spans = set()
    for event in example.get("event", []):
        # The trigger/event text itself
        if "text" in event:
            label_spans.add(event["text"])
        # All arguments
        for arg in event.get("args", []):
            label_spans.add(arg["text"])
            
    # Remove null values if any
    label_spans.discard("null")
    label_spans = {s.strip() for s in label_spans if s and s.strip()}
    
    # print(f"Text ID: {example.get('text_id')}")
    # print(f"Original Text: {text}")
    # print(f"Labeled Spans: {label_spans}")
    
    # Call the splitter
    generated_phrases = set(find_valid_phrases_list(text))#, max_chars_in_phrase=128))
    
    # print(f"Generierte Phrasen: {generated_phrases}")
    
    # Check coverage
    missing = [span for span in label_spans if span not in generated_phrases]
    
    if missing:
        missing_spans_report.append({
            "text_id": example.get("text_id"),
            "text": text,
            "missing": missing
        })
    
    results.append({
        "text_id": example.get("text_id"),
        "total_labels": len(label_spans),
        "missing_count": len(missing)
    })

total_examples = len(test_data)
failed_examples_count = len(missing_spans_report)
total_spans = sum(r["total_labels"] for r in results)
total_missing_spans = sum(r["missing_count"] for r in results)

print(f"Total Beispiele: {total_examples}")
print(f"Beispiele mit fehlenden Spans: {failed_examples_count}")
print(f"Total Label Spans: {total_spans}")
print(f"Total fehlende Spans: {total_missing_spans}")


Total Beispiele: 968
Beispiele mit fehlenden Spans: 0
Total Label Spans: 5291
Total fehlende Spans: 0


In [19]:
missing_spans_report

[]

In [20]:
if missing_spans_report:
    print("\n--- Fehlende Spans Details (Top 5) ---\n")
    for failure in missing_spans_report[:5]:
        print(f"ID: {failure['text_id']}")
        print(f"Text: {failure['text']}")
        print(f"Fehlende Spans: {failure['missing']}")
        print("-" * 20)
else:
    print("\nPerfekt! Alle Label-Spans wurden vom Splitter gefunden.")



Perfekt! Alle Label-Spans wurden vom Splitter gefunden.
